# FinReasoning AI Gradio Demo — vLLM + Adapters + Tools

This notebook launches a Gradio financial QA demo backed by the **vLLM inference engine**.
Supported features:
- **Configurable base model** (`BASE_MODEL_ID`) and optional **LoRA adapter** path (`ADAPTER_PATH`)
- Four inference modes: *Direct Answer*, *Chain-of-Thought*, *Self-Consistency (N=8)*, and **Tool-Augmented**
- Tool-Augmented mode calls `arithmetic` and `compound_growth_rate` tools via `<tool_call>` blocks
- Live benchmark metrics: TTFT, total latency, token throughput, peak VRAM

## Setup

Mount Drive, infer workspace, clone/pull repo, and install minimal dependencies.

In [ ]:
from pathlib import Path
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_FALLBACK = '/content/drive/MyDrive/FinReasoningAI'

def _infer_notebook_workspace():
    """Parent folder of FinReasoningAI_Colab.ipynb on Drive (depth-limited, quick)."""
    root = Path('/content/drive/MyDrive')
    if not root.is_dir():
        return None
    candidates = []
    if (root / 'FinReasoningAI_Colab.ipynb').is_file():
        candidates.append(root.resolve())
    for child in sorted(root.iterdir()):
        if not child.is_dir():
            continue
        if (child / 'FinReasoningAI_Colab.ipynb').is_file():
            candidates.append(child.resolve())
        nested = child / 'FinReasoningAI'
        if nested.is_dir() and (nested / 'FinReasoningAI_Colab.ipynb').is_file():
            candidates.append(nested.resolve())
    uniq = []
    seen = set()
    for c in candidates:
        s = str(c)
        if s not in seen:
            seen.add(s)
            uniq.append(c)
    if len(uniq) == 1:
        return str(uniq[0])
    if len(uniq) > 1:
        print('[WARN] Multiple FinReasoningAI_Colab.ipynb paths on Drive; using DRIVE_FALLBACK.')
    return None

DRIVE_BASE = _infer_notebook_workspace() or DRIVE_FALLBACK
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive workspace: {DRIVE_BASE}')

In [ ]:
REPO_URL = 'https://github.com/juankim834/FinReasoningAI.git'

import os
import sys

WORKSPACE = DRIVE_BASE
if os.path.isdir(os.path.join(WORKSPACE, '.git')):
    PROJECT_DIR = WORKSPACE
else:
    PROJECT_DIR = os.path.join(WORKSPACE, 'FinReasoningAI')

if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    os.makedirs(WORKSPACE, exist_ok=True)
    print(f'Cloning {REPO_URL} -> {PROJECT_DIR}')
    get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
else:
    print(f'Repo already at {PROJECT_DIR}. Pulling latest...')
    get_ipython().system(f'cd {PROJECT_DIR} && git pull')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
"""
Install only required dependencies (no torch reinstall).
Hard requirement: bitsandbytes >= 0.44.0.
"""

import importlib.metadata
import subprocess
import sys

REQUIRED = {
    'transformers': '4.41.0',
    'peft': '0.10.0',
    'bitsandbytes': '0.44.0',
    'accelerate': '0.30.0',
    'gradio': '4.0.0',
    'vllm': '0.4.0',
}

def _parse(v):
    out = []
    for p in v.split('.'):
        if p.isdigit():
            out.append(int(p))
        else:
            break
    while len(out) < 3:
        out.append(0)
    return tuple(out[:3])

def _installed(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None

to_install = []
for pkg, min_ver in REQUIRED.items():
    cur = _installed(pkg)
    if cur is None or _parse(cur) < _parse(min_ver):
        to_install.append(f'{pkg}>={min_ver}')
        status = 'MISSING' if cur is None else f'upgrade {cur} -> >= {min_ver}'
    else:
        status = f'ok ({cur})'
    print(f'{pkg:<14} {status}')

if to_install:
    print(f'\nInstalling {len(to_install)} package(s): {to_install}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + to_install)
    print('Install complete.')
else:
    print('\nAll required packages already satisfy minimum versions.')

bnb_ver = importlib.metadata.version('bitsandbytes')
if _parse(bnb_ver) < _parse('0.44.0'):
    raise RuntimeError(
        f'bitsandbytes {bnb_ver} is installed but >= 0.44.0 is required.\n'
        "Fix: pip install -U 'bitsandbytes>=0.44.0' then Runtime > Restart session."
    )

import torch
print(f'\nPyTorch version (unchanged): {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}')
    print(f'VRAM: {props.total_memory / (1024**3):.1f} GB')

In [ ]:
# Ensure outputs are symlinked to Drive.
from pathlib import Path
import shutil
import os

def ensure_drive_symlink(rel_path: str):
    drive_path = Path(DRIVE_BASE) / rel_path
    local_path = Path(PROJECT_DIR) / rel_path
    drive_path.mkdir(parents=True, exist_ok=True)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.is_symlink():
        print(f'[OK] Symlink exists: {local_path} -> {local_path.resolve()}')
        return

    if local_path.exists():
        if local_path.is_dir():
            shutil.copytree(local_path, drive_path, dirs_exist_ok=True)
            shutil.rmtree(local_path)
        else:
            shutil.copy2(local_path, drive_path)
            local_path.unlink()

    os.symlink(drive_path, local_path)
    print(f'[OK] Linked: {local_path} -> {drive_path}')

for _rel in ['outputs/merged_model', 'outputs/sft_qlora']:
    ensure_drive_symlink(_rel)

In [ ]:
# Optional Hugging Face login with fallback.
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in using Colab secret HF_TOKEN.')
except Exception:
    print('HF_TOKEN secret not found. Falling back to interactive login...')
    login()

## Configuration

Edit the variables in this cell to select the **base model**, an optional **LoRA adapter**, and the **vLLM engine** settings.

| Variable | Description |
|---|---|
| `BASE_MODEL_ID` | HF model ID **or** local path to the merged model (e.g. `outputs/merged_model`) |
| `ADAPTER_PATH` | Path to a saved LoRA/QLoRA adapter directory, or `None` to skip |
| `ENABLE_LORA_ADAPTER` | Set `True` to load the adapter via vLLM's built-in LoRA support |
| `MAX_MODEL_LEN` | Maximum context window length (tokens) |
| `GPU_MEMORY_UTIL` | Fraction of GPU memory reserved for the vLLM KV cache (0.0–1.0) |
| `TENSOR_PARALLEL_SIZE` | Number of GPUs for tensor parallelism (1 for single-GPU Colab) |

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------------
# Base model: HF hub ID or local directory path
# ---------------------------------------------------------------------------
BASE_MODEL_ID: str = 'Qwen/Qwen2.5-14B-Instruct'

# Auto-detect a local merged model and prefer it over the hub ID
_merged = Path('outputs/merged_model')
if _merged.exists() and any(_merged.iterdir()):
    BASE_MODEL_ID = str(_merged)
    print(f'[AUTO] Local merged model detected: {BASE_MODEL_ID}')
else:
    print(f'[AUTO] No local merged model found. Using hub ID: {BASE_MODEL_ID}')

# ---------------------------------------------------------------------------
# LoRA / QLoRA adapter (set ENABLE_LORA_ADAPTER=True and provide ADAPTER_PATH)
# ---------------------------------------------------------------------------
ADAPTER_PATH: str | None = 'outputs/sft_qlora/final_adapter'  # set to None to disable
ENABLE_LORA_ADAPTER: bool = True

# Validate adapter path
if ENABLE_LORA_ADAPTER:
    if ADAPTER_PATH is None:
        print('[WARN] ENABLE_LORA_ADAPTER=True but ADAPTER_PATH is None. Adapter disabled.')
        ENABLE_LORA_ADAPTER = False
    elif not Path(ADAPTER_PATH).exists():
        print(f'[WARN] Adapter path not found: {ADAPTER_PATH}. Adapter disabled.')
        ENABLE_LORA_ADAPTER = False
    else:
        print(f'[OK] Adapter path: {ADAPTER_PATH}')
else:
    print('[INFO] LoRA adapter disabled.')

# ---------------------------------------------------------------------------
# vLLM engine settings
# ---------------------------------------------------------------------------
MAX_MODEL_LEN: int = 2048        # token context window
GPU_MEMORY_UTIL: float = 0.90    # fraction of VRAM for KV cache
TENSOR_PARALLEL_SIZE: int = 1    # increase for multi-GPU
VLLM_DTYPE: str = 'bfloat16'     # 'bfloat16' | 'float16' | 'auto'

print(f'\n--- vLLM config ---')
print(f'  Base model          : {BASE_MODEL_ID}')
print(f'  Adapter             : {ADAPTER_PATH if ENABLE_LORA_ADAPTER else "(none)"}')
print(f'  max_model_len       : {MAX_MODEL_LEN}')
print(f'  gpu_memory_util     : {GPU_MEMORY_UTIL}')
print(f'  tensor_parallel_size: {TENSOR_PARALLEL_SIZE}')
print(f'  dtype               : {VLLM_DTYPE}')

## Model Load

Initialise the vLLM inference engine with the configured model and optional LoRA adapter.
If `ENABLE_LORA_ADAPTER=True`, the engine is loaded with `enable_lora=True` and a
`LoRARequest` is attached to every generation call.

In [ ]:
import torch
from src.model.load_model import DEFAULT_MODEL_ID, load_vllm_model_and_tokenizer
import src.inference.generate as _gen_module

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU runtime is required for this demo.')

print(f'Loading vLLM inference engine from: {BASE_MODEL_ID}')
model, tokenizer = load_vllm_model_and_tokenizer(
    model_id=BASE_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    dtype=VLLM_DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    enable_lora=ENABLE_LORA_ADAPTER,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------------------------------------------------------------------------
# Build LoRARequest (used for every generate() call when an adapter is active)
# ---------------------------------------------------------------------------
LORA_REQUEST = None
if ENABLE_LORA_ADAPTER and ADAPTER_PATH:
    try:
        from vllm.lora.request import LoRARequest
        LORA_REQUEST = LoRARequest('finreasoning_adapter', 1, ADAPTER_PATH)
        print(f'[OK] LoRA adapter loaded: {ADAPTER_PATH}')
    except Exception as _lora_err:
        print(f'[WARN] Could not create LoRARequest: {_lora_err}. Running without adapter.')
        LORA_REQUEST = None

# ---------------------------------------------------------------------------
# Monkey-patch _vllm_generate_texts to transparently inject lora_request
# so that generate_answer(), generate_with_tools(), etc. all use the adapter.
# ---------------------------------------------------------------------------
def _vllm_generate_adapter(
    model,
    prompts,
    *,
    temperature: float = 0.0,
    top_p: float = 0.9,
    max_new_tokens: int = 256,
    n: int = 1,
):
    """vLLM generation wrapper that injects an optional LoRA adapter request."""
    sp = _gen_module._build_vllm_sampling_params(
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        n=n,
    )
    outputs = model.generate(prompts, sp, lora_request=LORA_REQUEST)
    return [[c.text for c in req.outputs] for req in outputs]

_gen_module._vllm_generate_texts = _vllm_generate_adapter
print('[OK] vLLM generation patched (lora_request support active).')

# Re-import top-level helpers so they pick up the patched module state
from src.inference.generate import (
    generate_answer,
    generate_with_tools,
    build_prompt,
)

print('[OK] vLLM inference engine ready.')

## Keep-Alive

Start a daemon thread to keep the Colab runtime active during the demo session.

In [ ]:
import threading
import time

def _keep_alive_loop():
    while True:
        time.sleep(60)

keep_alive_thread = threading.Thread(target=_keep_alive_loop, daemon=True)
keep_alive_thread.start()
print('[OK] Keep-alive daemon thread started.')

## Demo Launch

Build and launch the Gradio interface.

**Inference modes**

| Mode | Description |
|---|---|
| Direct Answer | Single-pass greedy generation, grounding check applied |
| Chain-of-Thought | Extended reasoning via `<think>` blocks before the final answer |
| Self-Consistency (N=8) | 8 stochastic samples; answer selected by median/majority vote |
| **Tool-Augmented** | Multi-turn loop calling `arithmetic` / `compound_growth_rate` tools via `<tool_call>` blocks |

In [ ]:
import json
import re
import time

import gradio as gr
import pandas as pd
import torch

THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

APPLE_CONTEXT_PLACEHOLDER = (
    'Apple 2022 10-K excerpt:\n'
    'Net sales were $394.3 billion in 2022 and $365.8 billion in 2021.\n'
    'Research and development expense was $26.3 billion in 2022.\n'
    'Operating cash flow was $122.2 billion and capital expenditures were $10.7 billion.'
)

EXAMPLES = [
    [
        "What was Apple's year-over-year revenue growth from 2021 to 2022?",
        'Net sales were $394.3 billion in 2022 and $365.8 billion in 2021.',
        'Direct Answer',
    ],
    [
        "What percentage of revenue did Apple spend on R&D in 2022?",
        'Revenue was $394.3 billion and R&D expense was $26.3 billion in 2022.',
        'Chain-of-Thought',
    ],
    [
        "Estimate Apple's 2022 free cash flow using provided values.",
        'Operating cash flow was $122.2 billion and capex was $10.7 billion.',
        'Self-Consistency (N=8)',
    ],
    [
        "Calculate the exact year-over-year revenue growth percentage from 2021 to 2022.",
        'Net sales were $394.3 billion in 2022 and $365.8 billion in 2021.',
        'Tool-Augmented',
    ],
]


def _format_tool_trace(tool_calls: list, full_output: str) -> str:
    """Format tool call trace for display in the Reasoning tab."""
    if not tool_calls:
        return (full_output or '(no output)') + '\n\n[WARNING: No tool calls were made]'
    lines = []
    for i, tc in enumerate(tool_calls, 1):
        lines.append(f'=== Tool Call {i}: {tc["name"]} ===')
        lines.append(f'Arguments : {json.dumps(tc["arguments"], indent=2)}')
        lines.append(f'Result    : {json.dumps(tc["result"])}')
        lines.append('')
    lines.append('--- Full generation output ---')
    lines.append(full_output or '')
    return '\n'.join(lines)


def _estimate_ttft_ms(question: str, context: str, use_cot: bool) -> float:
    """Measure time-to-first-token by generating exactly 1 token."""
    prompt = build_prompt(question=question, context=context, use_cot=use_cot, tokenizer=tokenizer)
    t0 = time.perf_counter()
    _vllm_generate_adapter(model=model, prompts=[prompt], temperature=0.0, max_new_tokens=1, n=1)
    return (time.perf_counter() - t0) * 1000.0


def _benchmark_table(
    ttft_ms: float,
    total_ms: float,
    tokens_generated: int,
    tps: float,
    peak_vram_mb: float,
) -> pd.DataFrame:
    rows = [
        ['Time to first token (ms)', f'{ttft_ms:.2f}'],
        ['Total generation time (ms)', f'{total_ms:.2f}'],
        ['Tokens generated', str(tokens_generated)],
        ['Tokens per second', f'{tps:.2f}'],
        ['Peak VRAM used (MB)', f'{peak_vram_mb:.2f}'],
    ]
    return pd.DataFrame(rows, columns=['Metric', 'Value'])


def run_finreasoning_demo(question: str, context: str, mode: str):
    question = (question or '').strip()
    context = (context or '').strip()

    if not question:
        return 'Please provide a question.', '', _benchmark_table(0, 0, 0, 0, 0)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    use_cot = mode == 'Chain-of-Thought'

    # TTFT is measured for non-tool modes (tool mode has multi-turn overhead)
    ttft_ms = 0.0
    if mode != 'Tool-Augmented':
        ttft_ms = _estimate_ttft_ms(question=question, context=context, use_cot=use_cot)

    start = time.perf_counter()
    reasoning_text = ''
    text_for_tokens = ''

    if mode == 'Direct Answer':
        answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            use_cot=False,
            grounding_check=True,
            temperature=0.0,
        )
        reasoning_text = '(Chain-of-Thought not enabled for this mode)'
        text_for_tokens = answer

    elif mode == 'Chain-of-Thought':
        answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            use_cot=True,
            max_new_tokens=400,
            grounding_check=True,
            temperature=0.0,
        )
        # Capture the raw CoT trace (including <think> tags) for transparency
        _cot_prompt = build_prompt(
            question=question, context=context, use_cot=True, tokenizer=tokenizer
        )
        _raw = _vllm_generate_adapter(
            model=model, prompts=[_cot_prompt], temperature=0.0, max_new_tokens=400, n=1
        )[0][0].strip()
        reasoning_text = _raw
        if '</think>' not in _raw and '<think>' not in _raw:
            reasoning_text += '\n\n[INFO: No explicit <think> tags emitted in this run]'
        text_for_tokens = _raw

    elif mode == 'Self-Consistency (N=8)':
        answer = generate_answer(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            use_cot=False,
            self_consistency_n=8,
            temperature=0.7,
            min_confidence=0.30,
            grounding_check=True,
        )
        reasoning_text = '(Self-consistency aggregates 8 samples; individual traces not shown)'
        text_for_tokens = answer

    elif mode == 'Tool-Augmented':
        result = generate_with_tools(
            model=model,
            tokenizer=tokenizer,
            question=question,
            context=context,
            max_new_tokens=512,
        )
        raw_answer = result.get('answer') or ''
        answer = raw_answer or 'No answer could be extracted from the tool loop.'
        if result.get('max_rounds_exceeded'):
            answer = f'[WARNING: Max tool rounds exceeded]\n{answer}'
        if result.get('tool_violation'):
            answer = f'[WARNING: No tools were called — check the system prompt]\n{answer}'
        reasoning_text = _format_tool_trace(result['tool_calls'], result['full_output'])
        text_for_tokens = result['full_output'] or answer

    else:
        raise ValueError(f'Unsupported mode: {mode}')

    total_ms = (time.perf_counter() - start) * 1000.0
    tokens_generated = len(tokenizer.encode(text_for_tokens, add_special_tokens=False))
    tps = tokens_generated / (total_ms / 1000.0) if total_ms > 0 else 0.0
    peak_vram_mb = (
        torch.cuda.max_memory_allocated() / (1024 ** 2)
        if torch.cuda.is_available()
        else 0.0
    )

    bench_df = _benchmark_table(ttft_ms, total_ms, tokens_generated, tps, peak_vram_mb)
    return answer, reasoning_text, bench_df


# ---------------------------------------------------------------------------
# Build status line shown in the Gradio header
# ---------------------------------------------------------------------------
_model_label = BASE_MODEL_ID
_adapter_label = f'Adapter: `{ADAPTER_PATH}`' if LORA_REQUEST is not None else 'No adapter'

with gr.Blocks(title='FinReasoning AI — Financial QA Demo') as demo:
    gr.Markdown('# FinReasoning AI — Financial QA Demo')
    gr.Markdown(
        f'**Engine:** vLLM &nbsp;|&nbsp; '
        f'**Model:** `{_model_label}` &nbsp;|&nbsp; '
        f'{_adapter_label}'
    )

    with gr.Row():
        question_input = gr.Textbox(
            label='Question',
            placeholder='Enter your financial question...',
        )

    context_input = gr.Textbox(
        label='Financial Context',
        lines=8,
        placeholder=APPLE_CONTEXT_PLACEHOLDER,
    )

    mode_input = gr.Radio(
        choices=['Direct Answer', 'Chain-of-Thought', 'Self-Consistency (N=8)', 'Tool-Augmented'],
        value='Direct Answer',
        label='Inference Mode',
    )

    gr.Markdown(
        '_**Tool-Augmented** mode forces the model to call `arithmetic` or `compound_growth_rate`'
        ' tools before emitting a final numeric answer._'
    )

    run_btn = gr.Button('Run Inference', variant='primary')

    with gr.Tabs():
        with gr.Tab('Answer'):
            answer_output = gr.Textbox(label='Final Answer', lines=6)
        with gr.Tab('Reasoning / Tool Trace'):
            cot_output = gr.Textbox(
                label='Chain-of-Thought trace  |  Tool-Augmented call log',
                lines=18,
            )
        with gr.Tab('Efficiency Benchmark'):
            bench_output = gr.Dataframe(
                headers=['Metric', 'Value'],
                datatype=['str', 'str'],
                row_count=5,
                col_count=(2, 'fixed'),
                wrap=True,
                label='Benchmark',
            )

    run_btn.click(
        fn=run_finreasoning_demo,
        inputs=[question_input, context_input, mode_input],
        outputs=[answer_output, cot_output, bench_output],
    )

    gr.Examples(
        examples=EXAMPLES,
        inputs=[question_input, context_input, mode_input],
        outputs=[answer_output, cot_output, bench_output],
        fn=run_finreasoning_demo,
        cache_examples=False,
    )

demo.launch(share=True, debug=False)